# Day 6 - Snowflake
## Customer Support Analytics

Snowflake is the leading cloud data platform. It's where many organizations already store their operational and analytical data - data warehouse exports, CRM records, support tickets and clickstream events. Snowflake's native `VECTOR` data type and `VECTOR_COSINE_SIMILARITY` function add vector similarity search directly to that data, without moving it to a separate vector database.

**When would you reach for this?**
- Your data is already in Snowflake
- You want semantic search without introducing a new system
- Your audience is data engineers and analysts already working in SQL

**The use case:** A customer support analytics system that finds similar past support tickets and surfaces resolution patterns. Support teams can describe an issue in natural language and find the most relevant historical tickets, filtered by category, priority or status.

## 1. Setup

### Prerequisites

- A Snowflake free trial account
- Ollama running locally with the `all-minilm` model pulled
- Python 3.12 with a virtual environment

### Set Environment Variables

```bash
export SNOWFLAKE_ACCOUNT="your-account-identifier"
export SNOWFLAKE_USER="your-username"
export SNOWFLAKE_PASSWORD="your-password"
```

### Install Python Dependencies

In [1]:
%pip install ollama==0.6.2 \
             pandas==3.0.3 \
             pyarrow==18.1.0 \
             snowflake-connector-python==3.15.0 \
             tqdm==4.67.1 --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import ollama
import os
import pandas as pd
import random
import snowflake.connector

from snowflake.connector.pandas_tools import write_pandas
from tqdm.notebook import tqdm

### Configuration

In [3]:
SNOWFLAKE_ACCOUNT   = os.environ["SNOWFLAKE_ACCOUNT"]
SNOWFLAKE_USER      = os.environ["SNOWFLAKE_USER"]
SNOWFLAKE_PASSWORD  = os.environ["SNOWFLAKE_PASSWORD"]
SNOWFLAKE_DATABASE  = "SUPPORT_DB"
SNOWFLAKE_SCHEMA    = "SUPPORT_SCHEMA"
SNOWFLAKE_WAREHOUSE = "SUPPORT_WH"
SNOWFLAKE_ROLE      = "ACCOUNTADMIN"
TABLE_NAME          = "SUPPORT_TICKETS"
LLM_EMBEDDING       = "all-minilm"
NUM_TICKETS         = 200
RANDOM_SEED         = 42

> **Note:** `NUM_TICKETS` controls the size of the generated dataset. 200 is the recommended default. Embedding generation runs locally via Ollama and is single-threaded. At 200 tickets the notebook runs comfortably. Larger values will work but will take proportionally longer.

### Verify Ollama is Running

In [4]:
ollama_ready = False

try:
    models = ollama.list()
    model_names = [m.model for m in models.models]
    assert any(LLM_EMBEDDING in m for m in model_names)
    print(f"Model '{LLM_EMBEDDING}' is ready.")
    ollama_ready = True
except ConnectionError:
    print("ERROR: Ollama is not running. Start it with: ollama serve")
except AssertionError:
    print(f"ERROR: Model not found. Run: ollama pull {LLM_EMBEDDING}")

Model 'all-minilm' is ready.


In [5]:
assert ollama_ready, "Please fix the Ollama issue above before continuing."

### Determine Embedding Dimensions

In [6]:
def get_embedding(text: str) -> list:
    response = ollama.embeddings(model = LLM_EMBEDDING, prompt = text)
    return response["embedding"]

test_embedding = get_embedding("customer support ticket about billing issue")
EMBEDDING_DIMS = len(test_embedding)
print(f"Embedding dimensions: {EMBEDDING_DIMS}")

Embedding dimensions: 384


### Connect to Snowflake

In [7]:
conn = snowflake.connector.connect(
    account  = SNOWFLAKE_ACCOUNT,
    user     = SNOWFLAKE_USER,
    password = SNOWFLAKE_PASSWORD,
    role     = SNOWFLAKE_ROLE,
)

cursor = conn.cursor()

print(f"Connected to Snowflake: {conn.account}")

Connected to Snowflake: TYXQOKN-BX87004


### Create Database, Schema and Warehouse

In [8]:
cursor.execute(f"CREATE DATABASE IF NOT EXISTS {SNOWFLAKE_DATABASE}")
cursor.execute(f"USE DATABASE {SNOWFLAKE_DATABASE}")
cursor.execute(f"CREATE SCHEMA IF NOT EXISTS {SNOWFLAKE_SCHEMA}")
cursor.execute(f"USE SCHEMA {SNOWFLAKE_SCHEMA}")
cursor.execute(f"""
    CREATE WAREHOUSE IF NOT EXISTS {SNOWFLAKE_WAREHOUSE}
    WITH WAREHOUSE_SIZE = 'X-SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME  = TRUE
""")

cursor.execute(f"USE WAREHOUSE {SNOWFLAKE_WAREHOUSE}")

print(f"Database:  {SNOWFLAKE_DATABASE}")
print(f"Schema:    {SNOWFLAKE_SCHEMA}")
print(f"Warehouse: {SNOWFLAKE_WAREHOUSE}")

Database:  SUPPORT_DB
Schema:    SUPPORT_SCHEMA
Warehouse: SUPPORT_WH


## 2. The Dataset

We generate synthetic customer support tickets from pools of categories, priorities, products, statuses and description templates. Each ticket has a free-text description and a resolution note.

The `DESCRIPTION` field is what we embed and search over. The structured fields - `CATEGORY`, `PRIORITY`, `STATUS` and `PRODUCT` - serve as SQL filters.

In [9]:
random.seed(RANDOM_SEED)

CATEGORIES = [
    "Billing", "Technical", "Account", "Shipping", "Returns",
    "Product", "Security", "Performance", "Integration", "Other",
]

PRIORITIES = ["Low", "Medium", "High", "Critical"]

STATUSES = ["Open", "In Progress", "Resolved", "Closed"]

FICTIONAL_PRODUCTS = [
    "CloudSync Pro", "DataVault", "StreamLine", "SecurePortal",
    "AnalyticsDash", "ConnectAPI", "FlowBuilder", "ReportHub",
    "TaskMaster", "InsightEngine",
]

DESCRIPTION_TEMPLATES = {
    "Billing": [
        "Customer was charged twice for the same subscription period and is requesting a refund for the duplicate charge.",
        "Invoice amount does not match the quoted price. Customer is disputing the difference and requesting a corrected invoice.",
        "Customer's payment method was declined during renewal. Subscription has lapsed and customer cannot access the service.",
        "Customer received an unexpected charge on their account and cannot identify what it relates to.",
        "Customer is requesting a refund after cancelling their subscription within the cooling-off period.",
    ],
    "Technical": [
        "Customer reports the application is crashing on startup after the latest update was applied to their system.",
        "API calls are returning 500 errors intermittently. Customer has provided request IDs and timestamps for investigation.",
        "Data export feature is not generating the file. The download button appears to do nothing when clicked.",
        "Customer is experiencing slow load times on the dashboard. Pages are taking over 30 seconds to render.",
        "Integration with a third-party service stopped working after a configuration change on the customer's side.",
    ],
    "Account": [
        "Customer has forgotten their password and is not receiving the password reset email in their inbox.",
        "Customer needs to transfer account ownership to a new administrator following a staff change.",
        "Two-factor authentication is locked and customer cannot access their account from a new device.",
        "Customer wants to merge two separate accounts into a single account with combined data.",
        "Customer's account was suspended and they have not received any explanation or notification.",
    ],
    "Shipping": [
        "Order placed five days ago has not shipped. Tracking number shows no movement since label creation.",
        "Package was marked as delivered but customer has not received it. Requesting investigation or replacement.",
        "Customer received the wrong item in their order and needs the correct product sent urgently.",
        "Shipping address was entered incorrectly at checkout. Customer needs the address updated before dispatch.",
        "Package arrived damaged. Customer has sent photos and is requesting a replacement or refund.",
    ],
    "Returns": [
        "Customer wants to return a product purchased last week that does not meet their requirements.",
        "Return label was never received after the return was approved. Customer is following up.",
        "Refund for a returned item has not appeared after three weeks. Customer is requesting an update.",
        "Customer received a partial refund but was expecting a full refund per the stated returns policy.",
        "Customer wants to exchange a product for a different size rather than returning it for a refund.",
    ],
    "Product": [
        "Customer is requesting a feature that would allow bulk export of data in CSV format.",
        "A specific feature that was available in the previous version appears to have been removed in the latest release.",
        "Customer is asking for guidance on how to configure advanced settings that are not documented.",
        "Customer has found what appears to be a bug where saving a form clears previously entered data.",
        "Customer is requesting an integration with a tool they use daily that is not currently supported.",
    ],
    "Security": [
        "Customer suspects their account has been accessed by an unauthorized party and is requesting an audit.",
        "Customer received a phishing email appearing to come from our domain and wants to report it.",
        "Customer needs to know what data is stored about them for compliance with a regulatory request.",
        "Customer's API key was accidentally exposed in a public repository and needs to be revoked immediately.",
        "Customer is requesting a security review before deploying the product in a regulated environment.",
    ],
    "Performance": [
        "Customer reports that query execution times have increased significantly since last week's update.",
        "Scheduled reports that used to run in minutes are now timing out before completing.",
        "Customer is experiencing high memory usage that is causing their application to become unresponsive.",
        "Dashboard charts are loading slowly when the date range exceeds 90 days of data.",
        "Bulk data import is failing halfway through with a timeout error on large files.",
    ],
    "Integration": [
        "Webhook events are not being delivered to the customer's endpoint despite correct configuration.",
        "OAuth flow is failing at the authorization step when connecting to a third-party application.",
        "Data synced from the CRM is appearing with incorrect field mappings in the dashboard.",
        "Customer's scheduled sync job stopped running after the API version was updated.",
        "Customer needs help setting up a new integration that is not covered in the existing documentation.",
    ],
    "Other": [
        "Customer has a general enquiry about the product roadmap and upcoming features.",
        "Customer is requesting training resources for new team members joining their organization.",
        "Customer wants to provide feedback about their onboarding experience.",
        "Customer is asking about volume pricing for expanding their subscription to additional users.",
        "Customer needs a formal quote for renewal with updated user counts for their finance team.",
    ],
}

RESOLUTION_TEMPLATES = {
    "Billing":      "Billing team investigated and issued a credit to the customer's account. Invoice corrected and resent.",
    "Technical":    "Engineering team identified the root cause and deployed a fix. Customer confirmed the issue is resolved.",
    "Account":      "Account team verified the customer's identity and restored access. Security review completed.",
    "Shipping":     "Logistics team investigated the shipment. Replacement dispatched with expedited delivery.",
    "Returns":      "Returns processed and refund issued to the original payment method within 5 business days.",
    "Product":      "Product team reviewed the request and added it to the backlog. Customer notified of expected timeline.",
    "Security":     "Security team conducted an audit. No unauthorized access confirmed. API key revoked and reissued.",
    "Performance":  "Infrastructure team identified and resolved the performance bottleneck. Query times returned to normal.",
    "Integration":  "Integration team reviewed the configuration and corrected the field mappings. Sync confirmed working.",
    "Other":        "Customer query addressed by the relevant team. Follow-up scheduled if required.",
}

def generate_ticket(ticket_id: int) -> dict:
    category = random.choice(CATEGORIES)
    status = random.choice(STATUSES)
    return {
        "TICKET_ID":   f"TKT{ticket_id:05d}",
        "CATEGORY":    category,
        "PRIORITY":    random.choice(PRIORITIES),
        "STATUS":      status,
        "PRODUCT":     random.choice(FICTIONAL_PRODUCTS),
        "DESCRIPTION": random.choice(DESCRIPTION_TEMPLATES[category]),
        "RESOLUTION":  RESOLUTION_TEMPLATES[category] if status in ["Resolved", "Closed"] else None,
    }

tickets = [generate_ticket(i) for i in range(NUM_TICKETS)]
df = pd.DataFrame(tickets)

print(f"Generated {len(df)} support tickets")
print(f"\nCategory distribution:")
print(df["CATEGORY"].value_counts().to_string())
df.head()

Generated 200 support tickets

Category distribution:
CATEGORY
Shipping       32
Integration    26
Security       22
Returns        21
Product        20
Other          20
Technical      19
Account        17
Billing        12
Performance    11


,TICKET_ID,CATEGORY,PRIORITY,STATUS,PRODUCT,DESCRIPTION,RESOLUTION
0,TKT00000,Technical,High,Open,SecurePortal,API calls are returning 500 errors intermitten...,NaN
1,TKT00001,Account,Low,Open,InsightEngine,Customer wants to merge two separate accounts ...,NaN
2,TKT00002,Billing,Low,Open,SecurePortal,Invoice amount does not match the quoted price...,NaN
3,TKT00003,Integration,Medium,Open,TaskMaster,Customer's scheduled sync job stopped running ...,NaN
4,TKT00004,Shipping,High,Closed,CloudSync Pro,Package was marked as delivered but customer h...,Logistics team investigated the shipment. Repl...


## 3. Generate Embeddings

We embed each ticket description locally using Ollama. The embedding is stored as a list and will be loaded into Snowflake as an `ARRAY` column, which we cast to the native `VECTOR` type at query time.

In [10]:
embeddings = []
for ticket in tqdm(tickets, desc = "Generating embeddings"):
    embeddings.append(get_embedding(ticket["DESCRIPTION"]))

df["EMBEDDING"] = [str(e) for e in embeddings]
print(f"Generated {len(embeddings)} embeddings.")

Generating embeddings:   0%|          | 0/200 [00:00<?, ?it/s]

Generated 200 embeddings.


## 4. Load Data into Snowflake

We create a table with a `VECTOR` column and load the tickets. The embedding is stored as a `VARCHAR` and cast to `VECTOR(FLOAT, {EMBEDDING_DIMS})` at query time.

In [11]:
cursor.execute(f"DROP TABLE IF EXISTS {TABLE_NAME}")

cursor.execute(f"""
    CREATE TABLE {TABLE_NAME} (
        TICKET_ID   VARCHAR(20),
        CATEGORY    VARCHAR(50),
        PRIORITY    VARCHAR(20),
        STATUS      VARCHAR(20),
        PRODUCT     VARCHAR(100),
        DESCRIPTION VARCHAR(1000),
        RESOLUTION  VARCHAR(1000),
        EMBEDDING   VARCHAR(16000)
    )
""")

success, num_chunks, num_rows, _ = write_pandas(
    conn = conn,
    df = df,
    table_name = TABLE_NAME,
    auto_create_table = False
)

cursor.execute(f"SELECT COUNT(*) FROM {TABLE_NAME}")
count = cursor.fetchone()[0]
print(f"Loaded {count} tickets into {TABLE_NAME}.")

Loaded 200 tickets into SUPPORT_TICKETS.


## 5. Semantic Search

We embed the user's query locally with Ollama and pass it to Snowflake as a SQL parameter. The query casts the stored embedding string to `VECTOR(FLOAT, N)` and uses `VECTOR_COSINE_SIMILARITY` to rank results.

All of this is standard SQL - no separate search service, no SDK, no API call to a managed endpoint.

In [12]:
def search_tickets(query: str, top_k: int = 5):
    query_embedding = get_embedding(query)
    query_vec_str = str(query_embedding)

    cursor.execute(f"""
        SELECT
            TICKET_ID,
            CATEGORY,
            PRIORITY,
            STATUS,
            PRODUCT,
            DESCRIPTION,
            RESOLUTION,
            VECTOR_COSINE_SIMILARITY(
                TRY_PARSE_JSON(EMBEDDING)::VECTOR(FLOAT, {EMBEDDING_DIMS}),
                TRY_PARSE_JSON(%s)::VECTOR(FLOAT, {EMBEDDING_DIMS})
            ) AS similarity
        FROM {TABLE_NAME}
        ORDER BY similarity DESC
        LIMIT {top_k}
    """, (query_vec_str,))

    results = cursor.fetchall()
    print(f"\nQuery: '{query}'\n")
    for r in results:
        ticket_id, category, priority, status, product, description, resolution, similarity = r
        print(f"  {ticket_id} | {category} | {priority} | {status}")
        print(f"  Product: {product}")
        print(f"  Similarity: {similarity:.3f}")
        print(f"  Description: {description}")
        if resolution:
            print(f"  Resolution: {resolution}")
        print()

In [13]:
search_tickets("customer cannot log in after forgetting their password")


Query: 'customer cannot log in after forgetting their password'

  TKT00121 | Account | Medium | Open
  Product: AnalyticsDash
  Similarity: 0.714
  Description: Customer has forgotten their password and is not receiving the password reset email in their inbox.

  TKT00088 | Account | Critical | Resolved
  Product: TaskMaster
  Similarity: 0.714
  Description: Customer has forgotten their password and is not receiving the password reset email in their inbox.
  Resolution: Account team verified the customer's identity and restored access. Security review completed.

  TKT00070 | Account | Medium | Closed
  Product: CloudSync Pro
  Similarity: 0.492
  Description: Two-factor authentication is locked and customer cannot access their account from a new device.
  Resolution: Account team verified the customer's identity and restored access. Security review completed.

  TKT00114 | Account | High | Open
  Product: FlowBuilder
  Similarity: 0.492
  Description: Two-factor authentication is l

In [14]:
search_tickets("application is slow and pages take a long time to load")


Query: 'application is slow and pages take a long time to load'

  TKT00155 | Technical | Low | In Progress
  Product: FlowBuilder
  Similarity: 0.697
  Description: Customer is experiencing slow load times on the dashboard. Pages are taking over 30 seconds to render.

  TKT00104 | Technical | Medium | In Progress
  Product: SecurePortal
  Similarity: 0.697
  Description: Customer is experiencing slow load times on the dashboard. Pages are taking over 30 seconds to render.

  TKT00016 | Technical | High | In Progress
  Product: SecurePortal
  Similarity: 0.697
  Description: Customer is experiencing slow load times on the dashboard. Pages are taking over 30 seconds to render.

  TKT00138 | Performance | Critical | Resolved
  Product: TaskMaster
  Similarity: 0.483
  Description: Customer is experiencing high memory usage that is causing their application to become unresponsive.
  Resolution: Infrastructure team identified and resolved the performance bottleneck. Query times returned t

In [15]:
search_tickets("unauthorized access to account and suspicious activity")


Query: 'unauthorized access to account and suspicious activity'

  TKT00162 | Security | Critical | Resolved
  Product: ReportHub
  Similarity: 0.703
  Description: Customer suspects their account has been accessed by an unauthorized party and is requesting an audit.
  Resolution: Security team conducted an audit. No unauthorized access confirmed. API key revoked and reissued.

  TKT00014 | Security | Medium | Resolved
  Product: ConnectAPI
  Similarity: 0.703
  Description: Customer suspects their account has been accessed by an unauthorized party and is requesting an audit.
  Resolution: Security team conducted an audit. No unauthorized access confirmed. API key revoked and reissued.

  TKT00047 | Billing | Critical | In Progress
  Product: CloudSync Pro
  Similarity: 0.467
  Description: Customer received an unexpected charge on their account and cannot identify what it relates to.

  TKT00146 | Billing | Critical | Resolved
  Product: ReportHub
  Similarity: 0.467
  Description: C

## 6. Filtered Search

Because the tickets live in a standard Snowflake table, filtering is just SQL. We add `WHERE` clauses to the similarity query - no separate filter index declaration needed, no special filter syntax. Any column in the table can be used as a filter.

In [16]:
def search_tickets_filtered(
    query: str,
    category: str = None,
    priority: str = None,
    status: str   = None,
    product: str  = None,
    top_k: int    = 5
):
    query_embedding = get_embedding(query)
    query_vec_str = str(query_embedding)

    filters = []
    if category: filters.append(f"CATEGORY = '{category}'")
    if priority: filters.append(f"PRIORITY = '{priority}'")
    if status: filters.append(f"STATUS = '{status}'")
    if product: filters.append(f"PRODUCT = '{product}'")

    where_clause = "WHERE " + " AND ".join(filters) if filters else ""

    cursor.execute(f"""
        SELECT
            TICKET_ID,
            CATEGORY,
            PRIORITY,
            STATUS,
            PRODUCT,
            DESCRIPTION,
            RESOLUTION,
            VECTOR_COSINE_SIMILARITY(
                TRY_PARSE_JSON(EMBEDDING)::VECTOR(FLOAT, {EMBEDDING_DIMS}),
                TRY_PARSE_JSON(%s)::VECTOR(FLOAT, {EMBEDDING_DIMS})
            ) AS similarity
        FROM {TABLE_NAME}
        {where_clause}
        ORDER BY similarity DESC
        LIMIT {top_k}
    """, (query_vec_str,))

    results = cursor.fetchall()
    label = f"query = '{query}'"
    if category: label += f", category = '{category}'"
    if priority: label += f", priority = '{priority}'"
    if status: label += f", status = '{status}'"
    if product: label += f", product = '{product}'"
    print(f"\n{label}\n")

    for r in results:
        ticket_id, category_r, priority_r, status_r, product_r, description, resolution, similarity = r
        print(f"  {ticket_id} | {category_r} | {priority_r} | {status_r}")
        print(f"  Product: {product_r}")
        print(f"  Similarity: {similarity:.3f}")
        print(f"  Description: {description}")
        if resolution:
            print(f"  Resolution: {resolution}")
        print()

In [17]:
# High priority billing issues
search_tickets_filtered(
    "customer charged incorrectly and requesting refund",
    category = "Billing",
    priority = "High"
)


query = 'customer charged incorrectly and requesting refund', category = 'Billing', priority = 'High'

  TKT00050 | Billing | High | Open
  Product: CloudSync Pro
  Similarity: 0.715
  Description: Customer was charged twice for the same subscription period and is requesting a refund for the duplicate charge.

  TKT00117 | Billing | High | Open
  Product: SecurePortal
  Similarity: 0.715
  Description: Customer was charged twice for the same subscription period and is requesting a refund for the duplicate charge.

  TKT00160 | Billing | High | In Progress
  Product: InsightEngine
  Similarity: 0.613
  Description: Customer is requesting a refund after cancelling their subscription within the cooling-off period.

  TKT00196 | Billing | High | Open
  Product: CloudSync Pro
  Similarity: 0.419
  Description: Customer's payment method was declined during renewal. Subscription has lapsed and customer cannot access the service.



In [18]:
# Resolved technical issues - useful for finding past resolutions
search_tickets_filtered(
    "API returning errors and integration not working",
    category = "Technical",
    status = "Resolved"
)


query = 'API returning errors and integration not working', category = 'Technical', status = 'Resolved'

  TKT00072 | Technical | Critical | Resolved
  Product: InsightEngine
  Similarity: 0.506
  Description: API calls are returning 500 errors intermittently. Customer has provided request IDs and timestamps for investigation.
  Resolution: Engineering team identified the root cause and deployed a fix. Customer confirmed the issue is resolved.

  TKT00034 | Technical | Low | Resolved
  Product: InsightEngine
  Similarity: 0.340
  Description: Integration with a third-party service stopped working after a configuration change on the customer's side.
  Resolution: Engineering team identified the root cause and deployed a fix. Customer confirmed the issue is resolved.

  TKT00028 | Technical | High | Resolved
  Product: SecurePortal
  Similarity: 0.121
  Description: Customer reports the application is crashing on startup after the latest update was applied to their system.
  Resolution:

In [19]:
# Critical open tickets across all categories
search_tickets_filtered(
    "account locked and cannot access the system",
    priority = "Critical",
    status = "Open"
)


query = 'account locked and cannot access the system', priority = 'Critical', status = 'Open'

  TKT00113 | Billing | Critical | Open
  Product: StreamLine
  Similarity: 0.288
  Description: Customer received an unexpected charge on their account and cannot identify what it relates to.

  TKT00075 | Account | Critical | Open
  Product: CloudSync Pro
  Similarity: 0.210
  Description: Customer wants to merge two separate accounts into a single account with combined data.

  TKT00041 | Technical | Critical | Open
  Product: ConnectAPI
  Similarity: 0.208
  Description: Customer reports the application is crashing on startup after the latest update was applied to their system.

  TKT00148 | Technical | Critical | Open
  Product: ReportHub
  Similarity: 0.208
  Description: Customer reports the application is crashing on startup after the latest update was applied to their system.

  TKT00021 | Security | Critical | Open
  Product: FlowBuilder
  Similarity: 0.183
  Description: Customer i

## 7. Analytics with SQL

Because the tickets live in a standard Snowflake table, we can run analytical SQL queries alongside the semantic search - no separate system needed. Here we summarize ticket volumes and resolution rates by category.

In [20]:
cursor.execute(f"""
    SELECT
        CATEGORY,
        COUNT(*) AS TOTAL_TICKETS,
        SUM(CASE WHEN STATUS IN ('Resolved', 'Closed') THEN 1 ELSE 0 END) AS RESOLVED,
        SUM(CASE WHEN PRIORITY = 'Critical' THEN 1 ELSE 0 END) AS CRITICAL,
        ROUND(
            100.0 * SUM(CASE WHEN STATUS IN ('Resolved', 'Closed') THEN 1 ELSE 0 END) / COUNT(*),
            1
        ) AS RESOLUTION_RATE_PCT
    FROM {TABLE_NAME}
    GROUP BY CATEGORY
    ORDER BY TOTAL_TICKETS DESC
""")

summary = cursor.fetch_pandas_all()

print("Ticket summary by category:\n")
print(summary.to_string(index = False))

Ticket summary by category:

   CATEGORY  TOTAL_TICKETS  RESOLVED  CRITICAL RESOLUTION_RATE_PCT
   Shipping             32        16         8                50.0
Integration             26         9         6                34.6
   Security             22        13        10                59.1
    Returns             21        13         4                61.9
      Other             20        10         5                50.0
    Product             20        13         7                65.0
  Technical             19         6         5                31.6
    Account             17        11         7                64.7
    Billing             12         2         4                16.7
Performance             11         5         1                45.5


## Cleanup

In [21]:
cursor.execute(f"DROP TABLE IF EXISTS {TABLE_NAME}")
print(f"Table '{TABLE_NAME}' dropped.")

cursor.execute(f"DROP WAREHOUSE IF EXISTS {SNOWFLAKE_WAREHOUSE}")
print(f"Warehouse '{SNOWFLAKE_WAREHOUSE}' dropped.")

cursor.close()
conn.close()
print("Connection closed.")

Table 'SUPPORT_TICKETS' dropped.
Warehouse 'SUPPORT_WH' dropped.
Connection closed.
